# 📊 Preparação de Dados da Mega-Sena (Versão Simplificada)

## Estrutura Simplificada:
- 🔢 **Input (X)**: Apenas sequência de jogos em ordem `[[jogo1], [jogo2], ..., [jogo100]]`
- 🎯 **Target (y)**: Próximo jogo codificado em one-hot (60 posições)

## Divisão:
- 70% Treino | 15% Validação | 15% Teste

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')

# Configurações
WINDOW_SIZE = 500     # Jogos anteriores para análise
TOTAL_NUMBERS = 60    # Mega-sena: 1 a 60
NUMBERS_DRAWN = 6     # Quantidade de bolas sorteadas

print("📊 Preparação de Dados da Mega-Sena (Versão Simplificada)")
print(f"   WINDOW_SIZE: {WINDOW_SIZE}")
print(f"   TOTAL_NUMBERS: {TOTAL_NUMBERS}")

📊 Preparação de Dados da Mega-Sena (Versão Simplificada)
   WINDOW_SIZE: 500
   TOTAL_NUMBERS: 60


---
## 1. 📥 Carregamento dos Dados

In [20]:
try:
    df = pd.read_excel('Mega-Sena.xlsx')
    print("✅ Dados carregados com sucesso!")
    print(f"   Linhas: {len(df):,}")
    print(f"   Colunas: {df.columns.tolist()[:10]}...")
except FileNotFoundError:
    print("❌ Erro: Arquivo 'Mega-Sena.xlsx' não encontrado.")
except Exception as e:
    print(f"❌ Erro ao ler arquivo: {e}")

✅ Dados carregados com sucesso!
   Linhas: 2,954
   Colunas: ['Concurso', 'Data do Sorteio', 'Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6', 'Ganhadores 6 acertos', 'Cidade / UF']...


In [21]:
# Identificar colunas de bolas
ball_columns = []
for col in df.columns:
    if 'bola' in col.lower() or 'dezena' in col.lower():
        ball_columns.append(col)

# Se não encontrou por nome, tenta por range
if len(ball_columns) != 6:
    ball_columns = []
    for col in df.columns:
        numeric_col = pd.to_numeric(df[col], errors='coerce')
        if numeric_col.notna().sum() > 0:
            if numeric_col.min() >= 1 and numeric_col.max() <= 60:
                ball_columns.append(col)
    ball_columns = ball_columns[:6]

print(f"✅ Colunas de sorteio: {ball_columns}")

# Extrair e ordenar
data_balls = df[ball_columns].dropna().astype(int)
data_raw = np.sort(data_balls.values, axis=1)
print(f"✅ Shape dos dados: {data_raw.shape}")
print(f"   Total de concursos: {len(data_raw):,}")

✅ Colunas de sorteio: ['Bola1', 'Bola2', 'Bola3', 'Bola4', 'Bola5', 'Bola6']
✅ Shape dos dados: (2954, 6)
   Total de concursos: 2,954


---
## 2. 🛠️ Criação de Sequências Simples

**Estrutura:**
- `X`: Sequência de 100 jogos anteriores `(n_samples, 100, 6)`
- `y`: Próximo jogo codificado em one-hot `(n_samples, 60)`

In [22]:
def create_simple_sequences(data, window_size):
    """
    Cria sequências simples: apenas jogos em ordem.
    
    Args:
        data: array (n_games, 6) com todos os jogos
        window_size: tamanho da janela (100)
    
    Returns:
        X: (n_samples, window_size, 6) - sequência de jogos
        y: (n_samples, 6) - próximo jogo
    """
    X = []
    y = []
    
    for i in range(len(data) - window_size):
        # Janela de jogos anteriores
        window = data[i:i+window_size]
        # Próximo jogo (target)
        target = data[i+window_size]
        
        X.append(window)
        y.append(target)
    
    return np.array(X), np.array(y)


print("⏳ Criando sequências simples...")
X_games, y_raw = create_simple_sequences(data_raw, WINDOW_SIZE)

print(f"\n✅ Sequências criadas:")
print(f"   X_games (jogos anteriores): {X_games.shape}")
print(f"   y_raw (target): {y_raw.shape}")

⏳ Criando sequências simples...

✅ Sequências criadas:
   X_games (jogos anteriores): (2454, 500, 6)
   y_raw (target): (2454, 6)


In [23]:
# Visualizar um exemplo
idx = 0
print(f"\n📌 Exemplo de sequência #{idx}:")
print(f"\n   Últimos 5 jogos da janela:")
for i, game in enumerate(X_games[idx][-5:]):
    print(f"      Jogo {i+1}: {list(game)}")

print(f"\n   🎯 Próximo jogo (target): {list(y_raw[idx])}")


📌 Exemplo de sequência #0:

   Últimos 5 jogos da janela:
      Jogo 1: [10, 19, 20, 29, 41, 59]
      Jogo 2: [9, 13, 20, 32, 41, 58]
      Jogo 3: [2, 6, 28, 49, 50, 57]
      Jogo 4: [1, 4, 5, 11, 44, 59]
      Jogo 5: [21, 29, 35, 36, 38, 54]

   🎯 Próximo jogo (target): [16, 27, 28, 34, 39, 56]


---
## 3. 🎯 Codificação do Target (Multi-Hot Encoding)

In [24]:
# Multi-hot encoding do target
mlb = MultiLabelBinarizer(classes=range(1, TOTAL_NUMBERS + 1))
y_encoded = mlb.fit_transform(y_raw)

print(f"✅ Target codificado:")
print(f"   Shape: {y_encoded.shape}")
print(f"   Exemplo: {y_raw[0]} → vetor de {y_encoded.shape[1]} posições com {y_encoded[0].sum()} uns")

✅ Target codificado:
   Shape: (2454, 60)
   Exemplo: [16 27 28 34 39 56] → vetor de 60 posições com 6 uns


---
## 4. ✂️ Divisão Treino / Validação / Teste

In [25]:
total_samples = len(X_games)
train_idx = int(total_samples * 0.70)
val_idx = int(total_samples * 0.85)

# Divisão cronológica
X_train = X_games[:train_idx]
X_val = X_games[train_idx:val_idx]
X_test = X_games[val_idx:]

y_train = y_encoded[:train_idx]
y_val = y_encoded[train_idx:val_idx]
y_test = y_encoded[val_idx:]

print("="*70)
print("📊 DIVISÃO DOS DADOS")
print("="*70)
print(f"\n   Treino:    {len(X_train):,} amostras ({len(X_train)/total_samples*100:.1f}%)")
print(f"   Validação: {len(X_val):,} amostras ({len(X_val)/total_samples*100:.1f}%)")
print(f"   Teste:     {len(X_test):,} amostras ({len(X_test)/total_samples*100:.1f}%)")
print(f"\n   Total:     {total_samples:,} amostras")

📊 DIVISÃO DOS DADOS

   Treino:    1,717 amostras (70.0%)
   Validação: 368 amostras (15.0%)
   Teste:     369 amostras (15.0%)

   Total:     2,454 amostras


---
## 5. 💾 Salvar Dados Processados

---
## 4.5 🔝 Criação dos Vetores Top 10 e Bottom 10

Para cada amostra, identificamos:
- **Top 10**: Os 10 números que mais apareceram na janela (one-hot encoded)
- **Bottom 10**: Os 10 números que menos apareceram na janela (one-hot encoded)

Estes vetores ajudam o modelo a focar nos números "quentes" e "frios".


In [26]:
# ============================================================================
# CRIAÇÃO DOS VETORES DE TOP 10 E BOTTOM 10 NÚMEROS
# ============================================================================

def create_top_bottom_vectors(X_data):
    """
    Cria vetores one-hot para os 10 números mais e menos frequentes.
    
    Args:
        X_data: array de shape (n_samples, window_size, 6)
    
    Returns:
        top10_vectors: array (n_samples, 60) - one-hot dos 10 mais frequentes
        bottom10_vectors: array (n_samples, 60) - one-hot dos 10 menos frequentes
    """
    n_samples = X_data.shape[0]
    
    top10_vectors = np.zeros((n_samples, 60), dtype=np.float32)
    bottom10_vectors = np.zeros((n_samples, 60), dtype=np.float32)
    
    print(f"Criando vetores Top/Bottom 10 para {n_samples} amostras...")
    
    for i in range(n_samples):
        # Contar frequência de cada número
        freq = np.zeros(60)
        all_numbers = X_data[i].flatten().astype(int)
        
        for num in all_numbers:
            if 1 <= num <= 60:
                freq[num - 1] += 1
        
        # Ordenar índices por frequência
        sorted_indices = np.argsort(freq)
        
        # Bottom 10: 10 números menos frequentes
        bottom10_indices = sorted_indices[:10]
        bottom10_vectors[i, bottom10_indices] = 1.0
        
        # Top 10: 10 números mais frequentes
        top10_indices = sorted_indices[-10:]
        top10_vectors[i, top10_indices] = 1.0
    
    print(f"   Top10 shape: {top10_vectors.shape}")
    print(f"   Bottom10 shape: {bottom10_vectors.shape}")
    
    return top10_vectors, bottom10_vectors


print("="*60)
print("📊 CRIANDO VETORES TOP 10 E BOTTOM 10")
print("="*60)

print("\n🔹 Conjunto de Treino:")
X_train_top10, X_train_bottom10 = create_top_bottom_vectors(X_train)

print("\n🔹 Conjunto de Validação:")
X_val_top10, X_val_bottom10 = create_top_bottom_vectors(X_val)

print("\n🔹 Conjunto de Teste:")
X_test_top10, X_test_bottom10 = create_top_bottom_vectors(X_test)

print("\n✅ Vetores Top/Bottom 10 criados com sucesso!")


📊 CRIANDO VETORES TOP 10 E BOTTOM 10

🔹 Conjunto de Treino:
Criando vetores Top/Bottom 10 para 1717 amostras...
   Top10 shape: (1717, 60)
   Bottom10 shape: (1717, 60)

🔹 Conjunto de Validação:
Criando vetores Top/Bottom 10 para 368 amostras...
   Top10 shape: (368, 60)
   Bottom10 shape: (368, 60)

🔹 Conjunto de Teste:
Criando vetores Top/Bottom 10 para 369 amostras...
   Top10 shape: (369, 60)
   Bottom10 shape: (369, 60)

✅ Vetores Top/Bottom 10 criados com sucesso!


In [27]:
# Salvar arquivos .npy
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

# Salvar vetores Top/Bottom 10
np.save('X_train_top10.npy', X_train_top10)
np.save('X_train_bottom10.npy', X_train_bottom10)
np.save('X_val_top10.npy', X_val_top10)
np.save('X_val_bottom10.npy', X_val_bottom10)
np.save('X_test_top10.npy', X_test_top10)
np.save('X_test_bottom10.npy', X_test_bottom10)

print("\n✅ Dados salvos com sucesso!")
print("\nArquivos criados:")
print("   - X_train.npy, y_train.npy")
print("   - X_val.npy, y_val.npy")
print("   - X_test.npy, y_test.npy")
print("   - X_train_top10.npy, X_train_bottom10.npy")
print("   - X_val_top10.npy, X_val_bottom10.npy")
print("   - X_test_top10.npy, X_test_bottom10.npy")



✅ Dados salvos com sucesso!

Arquivos criados:
   - X_train.npy, y_train.npy
   - X_val.npy, y_val.npy
   - X_test.npy, y_test.npy
   - X_train_top10.npy, X_train_bottom10.npy
   - X_val_top10.npy, X_val_bottom10.npy
   - X_test_top10.npy, X_test_bottom10.npy


---
## 📋 Resumo Final

### Estrutura dos Dados:
- **X**: `(n_samples, 100, 6)` - Sequência de 100 jogos anteriores
- **y**: `(n_samples, 60)` - Próximo jogo codificado em one-hot

### Exemplo de uso no modelo:
```python
# Input: 100 jogos anteriores
# [[1, 5, 23, 34, 45, 56],
#  [2, 12, 25, 38, 41, 59],
#  ...
#  [3, 15, 28, 35, 48, 60]]
#
# Output: Probabilidades para cada número (1-60)
# [0.01, 0.95, 0.03, ..., 0.87]
```